# Chapter 7.8: Kernel Profiling — Nsight Compute, Nsight Systems

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Build a profiling workflow** for GPU kernel development
2. **Use Nsight Systems** for system-level profiling (timeline, overlap, overhead)
3. **Use Nsight Compute** for kernel-level profiling (roofline, occupancy, stalls)
4. **Interpret roofline analysis** to determine if a kernel is memory or compute bound
5. **Analyze occupancy** and understand its impact on performance
6. **Use `torch.profiler`** for PyTorch-level profiling
7. **Profile real LLM serving workloads** and identify bottlenecks
8. **Apply systematic optimization** based on profiling data

## Prerequisites

- Chapter 7.1 (CUDA Primer)
- NVIDIA Nsight Systems and Nsight Compute installed (optional for hands-on)
- `torch.profiler` available in PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part7/chapter_7.8_profiling.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part7/chapter_7.8_profiling.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Profiling Workflow Overview

### 1.1 The Three-Level Profiling Stack

```
Level 1: System-Level (Nsight Systems)
  → "Where is time spent? Are CPU and GPU well overlapped?"
  → Shows timeline of all kernels, memory transfers, CPU activity
  → Use FIRST to find bottleneck regions

Level 2: Kernel-Level (Nsight Compute)
  → "Why is this kernel slow? What limits it?"
  → Shows roofline, occupancy, memory throughput, warp stalls
  → Use SECOND on specific bottleneck kernels

Level 3: Python-Level (torch.profiler, py-spy)
  → "Where is Python overhead? Which PyTorch ops are called?"
  → Shows operator-level breakdown, memory allocation
  → Use for framework-level optimization
```

### 1.2 The Optimization Loop

```
1. Profile with Nsight Systems → Find the bottleneck kernel
2. Profile that kernel with Nsight Compute → Understand why it's slow
3. Apply optimization → Reduce the identified bottleneck
4. Re-profile → Verify improvement and find next bottleneck
5. Repeat until satisfied
```

**Golden rule**: Never optimize without profiling first. Your intuition about performance bottlenecks is usually wrong.

In [ ]:
# Check available profiling tools
import subprocess
import shutil

tools = {
    'nsys': 'Nsight Systems (system-level profiling)',
    'ncu': 'Nsight Compute (kernel-level profiling)',
    'nvprof': 'NVIDIA Visual Profiler (legacy)',
}

print("Profiling Tools Status:")
print("=" * 50)
for tool, description in tools.items():
    path = shutil.which(tool)
    status = f"FOUND at {path}" if path else "NOT FOUND"
    print(f"  {tool:<8} — {description}")
    print(f"           {status}")

# Check PyTorch profiler
try:
    from torch.profiler import profile, ProfilerActivity
    print(f"\n  torch.profiler — AVAILABLE")
except ImportError:
    print(f"\n  torch.profiler — NOT AVAILABLE (upgrade PyTorch)")

## 2. Nsight Systems: System-Level Profiling

### 2.1 What Nsight Systems Shows

Nsight Systems captures a **timeline** of all activity:

```
CPU Thread 1: [Python code][Launch kernel A][Wait for result][Process result]
CPU Thread 2:             [Data loading]         [Data loading]
GPU Stream 0:              [Kernel A  ][Kernel B][Kernel C      ]
GPU Stream 1:                [MemcpyH2D]    [MemcpyD2H]
```

### 2.2 Key Metrics

| Metric | What It Tells You |
|--------|------------------|
| **GPU utilization** | % of time GPU is busy (should be >90%) |
| **Kernel launch overhead** | Time between kernel launches (should be <10µs) |
| **CPU-GPU overlap** | Whether CPU work overlaps with GPU execution |
| **Memory transfer** | H2D/D2H copies and their overlap with compute |
| **Stream concurrency** | Whether multiple streams are used effectively |

### 2.3 How to Run

```bash
# Basic profiling
nsys profile -o output_report python my_script.py

# With CUDA API tracing
nsys profile --trace=cuda,nvtx -o report python my_script.py

# Limiting duration
nsys profile --duration=10 -o report python server.py

# View the report
nsys-ui report.nsys-rep  # GUI
nsys stats report.nsys-rep  # CLI summary
```

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesimport numpy as np# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, ax = plt.subplots(1, 1, figsize=(16, 6))ax.set_xlim(0, 50)ax.set_ylim(-0.5, 4.5)fig.patch.set_facecolor('white')ax.set_title('Mock Nsight Systems Timeline: LLM Serving', fontsize=14,             fontweight='bold', color=DARK_GRAY, pad=15)rows = {    'CPU (Python)': 3.5,    'CPU (Scheduler)': 2.5,    'GPU Stream 0': 1.5,    'GPU Stream 1': 0.5,}ax.set_yticks(list(rows.values()))ax.set_yticklabels(list(rows.keys()), fontsize=10, fontweight='bold')ax.set_xlabel('Time (ms)', fontsize=11)ax.grid(axis='x', alpha=0.3)# CPU Python barscpu_bars = [(0, 2, 'recv', LIGHT_BLUE), (3, 1, 'tok', LIGHT_GREEN),            (10, 2, 'recv', LIGHT_BLUE), (15, 1, 'tok', LIGHT_GREEN),            (25, 2, 'recv', LIGHT_BLUE), (30, 1, 'tok', LIGHT_GREEN)]for start, w, label, color in cpu_bars:    ax.barh(3.5, w, left=start, height=0.6, color=color, alpha=0.8, edgecolor='white', lw=1)    if w > 1:        ax.text(start+w/2, 3.5, label, ha='center', va='center', fontsize=7, color=DARK_GRAY)# CPU Scheduler barssched_bars = [(2, 3, 'schedule B0', BLUE), (12, 3, 'schedule B1', BLUE),              (27, 3, 'schedule B2', BLUE), (35, 3, 'schedule B3', BLUE)]for start, w, label, color in sched_bars:    ax.barh(2.5, w, left=start, height=0.6, color=color, alpha=0.85, edgecolor='white', lw=1)    ax.text(start+w/2, 2.5, label, ha='center', va='center', fontsize=7, color='white', fontweight='bold')# GPU Stream 0 (main compute)gpu_bars = [(5, 8, 'Attn Kernel', RED), (13, 4, 'FFN GEMM', PURPLE),            (17, 8, 'Attn Kernel', RED), (26, 4, 'FFN GEMM', PURPLE),            (31, 8, 'Attn Kernel', RED), (40, 4, 'FFN GEMM', PURPLE)]for start, w, label, color in gpu_bars:    ax.barh(1.5, w, left=start, height=0.6, color=color, alpha=0.85, edgecolor='white', lw=1)    ax.text(start+w/2, 1.5, label, ha='center', va='center', fontsize=7, color='white', fontweight='bold')# GPU Stream 1 (memory)mem_bars = [(4, 1, 'H2D', ORANGE), (14, 0.5, 'D2H', ORANGE),            (16, 1, 'H2D', ORANGE), (28, 0.5, 'D2H', ORANGE)]for start, w, label, color in mem_bars:    ax.barh(0.5, w, left=start, height=0.6, color=color, alpha=0.85, edgecolor='white', lw=1)    ax.text(start+w/2, 0.5, label, ha='center', va='center', fontsize=7, color='white', fontweight='bold')# Legendlegend_patches = [    mpatches.Patch(color=BLUE, label='Scheduling'),    mpatches.Patch(color=RED, label='Attention Kernel'),    mpatches.Patch(color=PURPLE, label='FFN GEMM'),    mpatches.Patch(color=ORANGE, label='Memory Transfer'),    mpatches.Patch(color=LIGHT_BLUE, label='Request I/O'),]ax.legend(handles=legend_patches, loc='upper right', fontsize=8, ncol=5)plt.tight_layout()plt.savefig("nsight_timeline.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

In [ ]:
# Simulate what Nsight Systems timeline analysis would reveal
import torch
import time

def analyze_kernel_timeline():
    """
    Simulate Nsight Systems analysis by measuring kernel launch patterns.
    """
    if not torch.cuda.is_available():
        print("GPU required")
        return
    
    N = 4096
    
    # Simulate a typical LLM decode step
    x = torch.randn(1, N, device='cuda', dtype=torch.float16)
    w1 = torch.randn(N, N, device='cuda', dtype=torch.float16)
    w2 = torch.randn(N, N, device='cuda', dtype=torch.float16)
    w3 = torch.randn(N, N * 4, device='cuda', dtype=torch.float16)
    w4 = torch.randn(N * 4, N, device='cuda', dtype=torch.float16)
    
    print("Simulated LLM Decode Timeline")
    print("=" * 60)
    
    operations = [
        ("RMSNorm", lambda: x / torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + 1e-6)),
        ("QKV Projection", lambda: torch.mm(x, w1)),
        ("Attention (simulated)", lambda: torch.softmax(torch.mm(x, w2), dim=-1)),
        ("Output Projection", lambda: torch.mm(x, w1)),
        ("RMSNorm", lambda: x / torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + 1e-6)),
        ("Gate Projection", lambda: torch.mm(x, w3)),
        ("Up Projection", lambda: torch.mm(x, w3)),
        ("SiLU + Mul", lambda: torch.nn.functional.silu(torch.randn(1, N*4, device='cuda'))),
        ("Down Projection", lambda: torch.mm(torch.randn(1, N*4, device='cuda', dtype=torch.float16), w4)),
    ]
    
    # Warmup
    for name, fn in operations:
        for _ in range(5):
            fn()
    
    # Measure each operation
    total_time = 0
    results = []
    
    for name, fn in operations:
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)
        
        start_event.record()
        for _ in range(100):
            fn()
        end_event.record()
        torch.cuda.synchronize()
        
        time_ms = start_event.elapsed_time(end_event) / 100
        total_time += time_ms
        results.append((name, time_ms))
    
    # Display timeline
    print(f"{'Operation':<25} {'Time (ms)':>10} {'% of Total':>12} {'Bar':>20}")
    print("-" * 70)
    
    for name, time_ms in results:
        pct = time_ms / total_time * 100
        bar = '#' * int(pct / 2)
        print(f"{name:<25} {time_ms:>10.3f} {pct:>11.1f}% {bar}")
    
    print("-" * 70)
    print(f"{'TOTAL':<25} {total_time:>10.3f}")
    print()
    print("Analysis:")
    
    # Find the biggest bottleneck
    biggest = max(results, key=lambda x: x[1])
    print(f"  Biggest bottleneck: {biggest[0]} ({biggest[1]:.3f} ms, {biggest[1]/total_time*100:.1f}%)")
    print(f"  → Profile this kernel with Nsight Compute for optimization")

analyze_kernel_timeline()

## 3. Nsight Compute: Kernel-Level Profiling

### 3.1 Key Sections

Nsight Compute provides detailed analysis of individual kernels:

| Section | What It Shows |
|---------|---------------|
| **GPU Speed of Light** | Overall throughput vs theoretical peak |
| **Roofline** | Compute vs memory bound analysis |
| **Compute Workload** | SM utilization, instruction mix |
| **Memory Workload** | Bandwidth utilization, L1/L2 hit rates |
| **Occupancy** | Theoretical vs achieved occupancy |
| **Warp State** | Where warps spend their time |
| **Source Correlation** | Map metrics to source code lines |

### 3.2 How to Run

```bash
# Profile all kernels (warning: very slow!)
ncu python my_script.py

# Profile a specific kernel
ncu --kernel-name "regex:softmax" python my_script.py

# Collect specific metrics
ncu --set full python my_script.py  # All metrics
ncu --set roofline python my_script.py  # Roofline analysis

# Save report
ncu -o report --set full python my_script.py
ncu-ui report.ncu-rep  # View in GUI
```

### 3.3 Roofline Analysis

The **roofline model** determines whether a kernel is limited by compute or memory:

```
Performance (FLOPS/s)
        │
Peak ── │─────────────────────────/
        │                       /
        │                     /
        │                   /  ← Memory Bound Region
        │                 /     (slope = bandwidth)
        │               /
        │             /
        │           /
        │         /
        │       /
        └──────┼──────────────────── Arithmetic Intensity (FLOPS/Byte)
               Ridge Point
```

**Ridge point** = Peak Compute / Peak Bandwidth

- A100: 312 TFLOPS / 2.0 TB/s = **156 FLOPS/Byte**
- If your kernel's AI < 156: **memory-bound** → optimize memory access
- If your kernel's AI > 156: **compute-bound** → optimize compute

In [ ]:
# Roofline analysis for common LLM operations
import numpy as np

def roofline_analysis():
    """
    Compute arithmetic intensity and classify operations.
    """
    # A100 specs
    peak_fp16 = 312e12  # FLOPS
    peak_bw = 2.0e12    # Bytes/s
    ridge_point = peak_fp16 / peak_bw
    
    print(f"A100 Roofline Analysis")
    print(f"Peak FP16: {peak_fp16/1e12:.0f} TFLOPS")
    print(f"Peak HBM BW: {peak_bw/1e12:.1f} TB/s")
    print(f"Ridge Point: {ridge_point:.0f} FLOPS/Byte")
    print("=" * 80)
    
    # Common LLM operations
    operations = [
        {
            "name": "GEMM (4096×4096, batch=256)",
            "flops": 2 * 256 * 4096 * 4096,
            "bytes": (256 * 4096 + 4096 * 4096 + 256 * 4096) * 2,  # A, B, C in FP16
        },
        {
            "name": "GEMM (4096×4096, batch=1)",
            "flops": 2 * 1 * 4096 * 4096,
            "bytes": (1 * 4096 + 4096 * 4096 + 1 * 4096) * 2,
        },
        {
            "name": "Softmax (seq=2048, heads=32)",
            "flops": 32 * 5 * 2048 * 2048,  # ~5 ops per element
            "bytes": 32 * 2 * 2048 * 2048 * 2,  # Read + write, FP16
        },
        {
            "name": "RMSNorm (batch=256, hidden=4096)",
            "flops": 256 * 4096 * 5,  # sq, sum, div, sqrt, mul
            "bytes": 256 * 4096 * 2 * 3,  # Read input, weight, write output
        },
        {
            "name": "SiLU-Mul (batch=256, FFN=11008)",
            "flops": 256 * 11008 * 4,  # sigmoid, mul, mul
            "bytes": 256 * 11008 * 2 * 3,  # Read gate, up, write output
        },
        {
            "name": "PagedAttention Decode (batch=32, seq=2048)",
            "flops": 32 * 32 * 2 * 2048 * 128,  # Q·K + weights·V per head
            "bytes": 32 * 32 * 2 * 2048 * 128 * 2,  # Load K, V per head
        },
        {
            "name": "RoPE (batch=256, heads=32, dim=128)",
            "flops": 256 * 32 * 128 * 6,  # sin, cos, mul, mul, add, sub
            "bytes": 256 * 32 * 128 * 2 * 4,  # Q, cos, sin, Q_out
        },
    ]
    
    print(f"{'Operation':<45} {'AI':>6} {'Bound':>10} {'Peak Eff':>10}")
    print("-" * 75)
    
    for op in operations:
        ai = op['flops'] / op['bytes']
        bound = "COMPUTE" if ai > ridge_point else "MEMORY"
        
        if bound == "MEMORY":
            achievable = ai * peak_bw  # Limited by bandwidth
        else:
            achievable = peak_fp16  # Limited by compute
        
        efficiency = achievable / peak_fp16 * 100
        
        print(f"{op['name']:<45} {ai:>5.1f} {bound:>10} {efficiency:>9.1f}%")
    
    print()
    print("Key insights:")
    print("  - Large-batch GEMM is compute-bound → optimize tensor core usage")
    print("  - Small-batch GEMM is memory-bound → optimize weight loading")
    print("  - Elementwise ops (RMSNorm, SiLU, RoPE) are always memory-bound")
    print("  - Decode attention is memory-bound → optimize KV cache loading")

roofline_analysis()

### 3.4 Occupancy Analysis

**Occupancy** = Active warps / Maximum warps per SM

Higher occupancy helps hide memory latency by having more warps ready to execute while others wait for data.

Occupancy is limited by:
1. **Registers per thread**: More registers → fewer threads → lower occupancy
2. **Shared memory per block**: More shared memory → fewer blocks → lower occupancy
3. **Block size**: Must be a multiple of warp size (32)

**Important**: Higher occupancy doesn't always mean better performance. If a kernel is compute-bound, the existing warps may already saturate the compute units.

In [ ]:
# Occupancy calculator
def calculate_occupancy(
    threads_per_block=256,
    registers_per_thread=32,
    shared_mem_per_block=0,
    # A100 SM specs
    max_threads_per_sm=2048,
    max_blocks_per_sm=32,
    max_registers_per_sm=65536,
    max_shared_mem_per_sm=164 * 1024,  # 164 KB
    warp_size=32,
):
    """
    Calculate theoretical occupancy for a kernel configuration.
    """
    warps_per_block = threads_per_block // warp_size
    max_warps_per_sm = max_threads_per_sm // warp_size
    
    # Limit 1: Thread count
    blocks_by_threads = max_threads_per_sm // threads_per_block
    
    # Limit 2: Block count
    blocks_by_blocks = max_blocks_per_sm
    
    # Limit 3: Registers
    regs_per_block = registers_per_thread * threads_per_block
    blocks_by_regs = max_registers_per_sm // regs_per_block if regs_per_block > 0 else max_blocks_per_sm
    
    # Limit 4: Shared memory
    blocks_by_smem = max_shared_mem_per_sm // shared_mem_per_block if shared_mem_per_block > 0 else max_blocks_per_sm
    
    # Actual blocks per SM = minimum of all limits
    active_blocks = min(blocks_by_threads, blocks_by_blocks, blocks_by_regs, blocks_by_smem)
    active_warps = active_blocks * warps_per_block
    occupancy = active_warps / max_warps_per_sm
    
    print(f"Kernel Configuration:")
    print(f"  Threads/block: {threads_per_block}")
    print(f"  Registers/thread: {registers_per_thread}")
    print(f"  Shared memory/block: {shared_mem_per_block/1024:.1f} KB")
    print()
    
    limiting = []
    print(f"Occupancy Limiters:")
    limits = [
        ("Thread count", blocks_by_threads),
        ("Block count", blocks_by_blocks),
        ("Registers", blocks_by_regs),
        ("Shared memory", blocks_by_smem),
    ]
    
    for name, limit in limits:
        is_limiting = limit == active_blocks
        marker = " ← LIMITING" if is_limiting else ""
        print(f"  {name:<15}: {limit} blocks/SM{marker}")
        if is_limiting:
            limiting.append(name)
    
    print()
    print(f"Result:")
    print(f"  Active blocks/SM: {active_blocks}")
    print(f"  Active warps/SM: {active_warps}/{max_warps_per_sm}")
    print(f"  Occupancy: {occupancy:.0%}")
    print(f"  Limited by: {', '.join(limiting)}")
    
    return occupancy


print("=" * 60)
print("Case 1: RMSNorm kernel (low register usage)")
print("=" * 60)
calculate_occupancy(threads_per_block=256, registers_per_thread=24, shared_mem_per_block=0)

print("\n" + "=" * 60)
print("Case 2: Matmul kernel (high register + shared memory)")
print("=" * 60)
calculate_occupancy(threads_per_block=256, registers_per_thread=64, shared_mem_per_block=32*1024)

print("\n" + "=" * 60)
print("Case 3: FlashAttention (heavy shared memory usage)")
print("=" * 60)
calculate_occupancy(threads_per_block=128, registers_per_thread=128, shared_mem_per_block=96*1024)

### 3.5 Warp Stall Reasons

When a warp cannot issue an instruction, it is **stalled**. Common stall reasons:

| Stall Reason | Meaning | Fix |
|-------------|---------|-----|
| **Long Scoreboard** | Waiting for global memory read | Better coalescing, more ILP |
| **Short Scoreboard** | Waiting for shared memory or special function | Reduce bank conflicts |
| **Wait** | Barrier synchronization | Reduce sync points |
| **Not Selected** | Other warps chosen (not a problem) | N/A |
| **MIO Throttle** | Memory I/O path saturated | Reduce memory operations |
| **Math Throttle** | Compute path saturated (good!) | N/A |

## 4. PyTorch Profiler

### 4.1 Basic Usage

```python
from torch.profiler import profile, ProfilerActivity, schedule

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    schedule=schedule(wait=1, warmup=1, active=3, repeat=1),
    on_trace_ready=torch.profiler.tensorboard_trace_handler('./logs'),
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
) as prof:
    for step in range(10):
        model(input)
        prof.step()
```

In [ ]:
import torch
from torch.profiler import profile, ProfilerActivity

def profile_llm_operations():
    """Profile common LLM operations using PyTorch profiler."""
    if not torch.cuda.is_available():
        print("GPU required")
        return
    
    # Setup
    batch, seq_len, hidden, heads = 32, 512, 4096, 32
    head_dim = hidden // heads
    ffn_hidden = 11008
    
    # Create tensors
    x = torch.randn(batch, seq_len, hidden, device='cuda', dtype=torch.float16)
    w_qkv = torch.randn(hidden, 3 * hidden, device='cuda', dtype=torch.float16)
    w_o = torch.randn(hidden, hidden, device='cuda', dtype=torch.float16)
    w_gate = torch.randn(hidden, ffn_hidden, device='cuda', dtype=torch.float16)
    w_up = torch.randn(hidden, ffn_hidden, device='cuda', dtype=torch.float16)
    w_down = torch.randn(ffn_hidden, hidden, device='cuda', dtype=torch.float16)
    norm_weight = torch.ones(hidden, device='cuda', dtype=torch.float16)
    
    # Profile
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=True,
    ) as prof:
        # Simulate one transformer layer
        for _ in range(3):  # 3 iterations for better statistics
            # RMSNorm
            x_flat = x.view(-1, hidden)
            rms = torch.sqrt(x_flat.float().pow(2).mean(-1, keepdim=True) + 1e-6)
            x_norm = (x_flat / rms * norm_weight).half()
            
            # QKV projection
            qkv = torch.mm(x_norm, w_qkv)
            
            # Attention (simplified)
            q, k, v = qkv.chunk(3, dim=-1)
            q = q.view(batch, seq_len, heads, head_dim).transpose(1, 2)
            k = k.view(batch, seq_len, heads, head_dim).transpose(1, 2)
            v = v.view(batch, seq_len, heads, head_dim).transpose(1, 2)
            attn = torch.nn.functional.scaled_dot_product_attention(q, k, v)
            attn = attn.transpose(1, 2).contiguous().view(-1, hidden)
            
            # Output projection
            attn_out = torch.mm(attn, w_o)
            
            # FFN
            gate = torch.mm(x_norm, w_gate)
            up = torch.mm(x_norm, w_up)
            activated = torch.nn.functional.silu(gate) * up
            down = torch.mm(activated, w_down)
    
    # Print results
    print(prof.key_averages().table(
        sort_by="cuda_time_total", 
        row_limit=15,
        max_name_column_width=40
    ))
    
    return prof

prof = profile_llm_operations()

## 5. Profiling a Real vLLM/SGLang Workload

### 5.1 Profiling vLLM

```bash
# System-level profiling with Nsight Systems
nsys profile --trace=cuda,nvtx \
    --output=vllm_profile \
    python -m vllm.entrypoints.openai.api_server \
    --model meta-llama/Llama-2-7b-hf \
    --max-model-len 2048

# Then send some requests and stop the profiler
```

### 5.2 What to Look For

| Observation | Meaning | Action |
|------------|---------|--------|
| GPU idle between kernels | Kernel launch overhead | Batch operations, use CUDA graphs |
| Small kernels dominate | Overhead from too many kernels | Fuse operations |
| One kernel takes >50% time | Clear bottleneck | Profile with Nsight Compute |
| CPU busy while GPU waits | Python overhead | Use async processing, C++ inference |
| Memory copies during compute | Suboptimal data movement | Use pinned memory, overlap copies |

In [ ]:
# Generate Nsight Systems profiling commands
def generate_profiling_commands():
    print("Profiling Commands for LLM Serving")
    print("=" * 60)
    
    print("\n1. System-level profiling (Nsight Systems):")
    print("─" * 50)
    print("# Profile the entire inference pipeline")
    print('nsys profile --trace=cuda,nvtx,cudnn,cublas \\')
    print('    --sample=cpu \\')
    print('    --output=llm_serving \\')
    print('    --force-overwrite=true \\')
    print('    python benchmark_serving.py')
    print()
    print('# View summary statistics')
    print('nsys stats llm_serving.nsys-rep')
    
    print("\n2. Kernel-level profiling (Nsight Compute):")
    print("─" * 50)
    print("# Profile attention kernels")
    print('ncu --kernel-name "regex:attention|paged" \\')
    print('    --set full \\')
    print('    --output=attention_analysis \\')
    print('    --target-processes all \\')
    print('    python benchmark_serving.py')
    print()
    print('# Profile GEMM kernels')
    print('ncu --kernel-name "regex:gemm|cutlass|sm80" \\')
    print('    --set roofline \\')
    print('    --output=gemm_analysis \\')
    print('    python benchmark_serving.py')
    
    print("\n3. PyTorch profiler with TensorBoard:")
    print("─" * 50)
    print("# In your Python code:")
    print('from torch.profiler import profile, ProfilerActivity, schedule')
    print()
    print('with profile(')
    print('    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],')
    print('    schedule=schedule(wait=2, warmup=2, active=6, repeat=1),')
    print('    on_trace_ready=torch.profiler.tensorboard_trace_handler("./logs"),')
    print('    record_shapes=True,')
    print('    profile_memory=True,')
    print('    with_stack=True,')
    print(') as prof:')
    print('    for step, batch in enumerate(dataloader):')
    print('        output = model(batch)')
    print('        prof.step()')
    print()
    print('# View in TensorBoard')
    print('tensorboard --logdir=./logs')

generate_profiling_commands()

## 6. Common Performance Issues and Fixes

### 6.1 Issue → Diagnosis → Fix Table

| Symptom | Nsight Metric | Root Cause | Fix |
|---------|--------------|------------|-----|
| Low GPU utilization | Timeline gaps | Kernel launch overhead | CUDA graphs, batching |
| Low bandwidth utilization | Memory throughput < 50% | Non-coalesced access | Reorder data layout |
| Low compute utilization | SM occupancy < 30% | Too many registers | Reduce register usage |
| High L1 miss rate | L1 hit rate < 50% | Working set too large | Use shared memory tiling |
| Bank conflicts | Shared memory efficiency < 80% | Stride access pattern | Pad shared memory |
| Warp divergence | Branch efficiency < 80% | Conditional logic in warp | Restructure conditions |

In [ ]:
# Demonstrate a kernel optimization workflow
import torch
import time

if torch.cuda.is_available():
    print("Optimization Workflow Example: Softmax Kernel")
    print("=" * 60)
    
    N, D = 4096, 4096
    x = torch.randn(N, D, device='cuda', dtype=torch.float16)
    
    # Version 1: Naive (3 separate ops)
    def softmax_v1(x):
        m = x.max(dim=-1, keepdim=True).values  # Op 1: max
        e = torch.exp(x - m)                     # Op 2: exp
        s = e.sum(dim=-1, keepdim=True)          # Op 3: sum
        return e / s                              # Op 4: div
    
    # Version 2: PyTorch built-in (fused internally)
    def softmax_v2(x):
        return torch.softmax(x, dim=-1)
    
    # Benchmark
    for name, fn in [("Naive (4 kernels)", softmax_v1), ("torch.softmax (fused)", softmax_v2)]:
        # Warmup
        for _ in range(10):
            fn(x)
        
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(100):
            fn(x)
        torch.cuda.synchronize()
        elapsed = (time.perf_counter() - start) / 100 * 1e6
        
        # Bandwidth
        bytes_accessed = 2 * N * D * 2  # Read + write, FP16
        bandwidth = bytes_accessed / (elapsed * 1e-6) / 1e9
        
        print(f"  {name:<25}: {elapsed:.0f} µs, {bandwidth:.0f} GB/s")
    
    print()
    print("Profiling insights:")
    print("  - Naive: 4 kernel launches, 4× memory reads/writes")
    print("  - Fused: 1 kernel launch, 1× memory read + 1× write")
    print("  - The difference is entirely due to memory traffic")
    print("  → Nsight Compute would show 'Long Scoreboard' stalls in naive version")

## 7. Building a Complete Profiling Workflow

### Step-by-Step Guide

```
Step 1: Baseline Measurement
  → Measure end-to-end latency and throughput
  → Record: tokens/sec, TTFT, TPOT

Step 2: System-Level Profile (Nsight Systems)
  → Identify the top-5 time-consuming kernels
  → Check for GPU idle time, launch overhead
  → Check CPU-GPU overlap

Step 3: Kernel-Level Profile (Nsight Compute)
  → For each bottleneck kernel:
     a. Check roofline: memory-bound or compute-bound?
     b. If memory-bound: check coalescing, L1/L2 hit rates
     c. If compute-bound: check occupancy, tensor core utilization
     d. Check warp stall reasons

Step 4: Optimize
  → Apply the appropriate fix (see issue table)
  → Re-profile to verify improvement

Step 5: Iterate
  → The bottleneck shifts after each optimization
  → Stop when you reach acceptable performance
```

In [ ]:
# Summary checklist
print("GPU Kernel Optimization Checklist")
print("=" * 60)
print()

checklist = [
    ("Memory Coalescing", "All global memory accesses are coalesced?"),
    ("Bank Conflicts", "No shared memory bank conflicts?"),
    ("Occupancy", "Sufficient occupancy to hide latency? (>50%)"),
    ("Register Spilling", "No register spilling to local memory?"),
    ("Warp Divergence", "Minimal branch divergence within warps?"),
    ("Kernel Fusion", "Adjacent memory-bound ops are fused?"),
    ("Tensor Cores", "Using tensor cores for matmul?"),
    ("Memory Reuse", "Data loaded to shared memory is reused?"),
    ("Launch Overhead", "Kernel launch overhead < computation time?"),
    ("CUDA Graphs", "Repetitive launch patterns use CUDA graphs?"),
    ("Precision", "Using lowest safe precision (FP16/BF16/FP8)?"),
    ("Async Operations", "Overlapping computation with data transfer?"),
]

for i, (area, question) in enumerate(checklist, 1):
    print(f"  [{' '}] {i:2d}. {area:<20} — {question}")

print()
print("Use this checklist when reviewing kernel performance.")
print("Each unchecked item is a potential optimization opportunity.")

## 8. Summary

| Tool | Level | Use Case | Key Output |
|------|-------|----------|------------|
| **Nsight Systems** | System | Find bottleneck kernels | Timeline, GPU utilization |
| **Nsight Compute** | Kernel | Understand why a kernel is slow | Roofline, occupancy, stalls |
| **torch.profiler** | Framework | Python/PyTorch overhead | Op-level breakdown |
| **py-spy** | Python | Pure Python bottlenecks | Python call graph |

### Key Principles

1. **Profile first, optimize second** — never guess where the bottleneck is
2. **Start at system level** — Nsight Systems before Nsight Compute
3. **Memory-bound vs compute-bound** — determines the optimization strategy
4. **The bottleneck shifts** — after fixing one issue, profile again
5. **Good enough is enough** — stop when you reach acceptable performance

## Exercises

### Exercise 1: Profile a Matmul Kernel

Using `torch.profiler`, profile matrix multiplications of different sizes and determine at what size the operation transitions from memory-bound to compute-bound.

### Exercise 2: Identify Fusion Opportunities

Profile a simple transformer layer and identify which adjacent operations could be fused. Estimate the expected speedup.

### Exercise 3: CUDA Graphs Impact

Measure the kernel launch overhead for a sequence of small kernels (e.g., 10 elementwise operations). Then wrap them in a CUDA graph and measure the improvement.

In [ ]:
# Exercise 3: CUDA Graphs demonstration
import torch
import time

if torch.cuda.is_available():
    N = 1024
    x = torch.randn(N, N, device='cuda')
    
    # Without CUDA graphs: many small kernel launches
    def run_ops_no_graph(x, n_ops=20):
        for _ in range(n_ops):
            x = x * 1.001  # Small elementwise operation
            x = x + 0.001
        return x
    
    # With CUDA graphs: capture and replay
    # Warmup
    s = torch.cuda.Stream()
    s.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(s):
        run_ops_no_graph(x)
    torch.cuda.current_stream().wait_stream(s)
    
    # Capture graph
    g = torch.cuda.CUDAGraph()
    x_static = x.clone()
    with torch.cuda.graph(g):
        y_static = run_ops_no_graph(x_static)
    
    # Benchmark: no graph
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(100):
        _ = run_ops_no_graph(x)
    torch.cuda.synchronize()
    no_graph_time = (time.perf_counter() - start) / 100 * 1e6
    
    # Benchmark: with graph
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(100):
        g.replay()
    torch.cuda.synchronize()
    graph_time = (time.perf_counter() - start) / 100 * 1e6
    
    print(f"20 small elementwise ops on {N}×{N} tensor:")
    print(f"  Without CUDA graph: {no_graph_time:.0f} µs")
    print(f"  With CUDA graph:    {graph_time:.0f} µs")
    print(f"  Speedup: {no_graph_time/graph_time:.2f}x")
    print()
    print("CUDA graphs eliminate kernel launch overhead by recording")
    print("the entire sequence once and replaying it as a single operation.")
    print("This is especially beneficial for small kernels with high launch overhead.")